# СИМА — Сервис рельефа: полная демонстрация

Ноутбук-демонстрация библиотеки **`sima-relief-service`** — сервисного слоя блока
«Анализ рельефа» из Q3-концепции. Покрывает весь конвейер: оценка материалов →
расчёт ЦМР (SMRF) → сглаживание → производные (уклон/экспозиция/TPI) → векторы
(горизонтали .shp, отметки высот .shp, TIN .dxf), с визуализацией промежуточных
результатов и верификацией против эталонов.

Структура и пути к данным переиспользованы из `sima_dsm_demo.ipynb`.

In [ ]:
import sys, os, warnings
from pathlib import Path

# --- подавление предупреждений среды (GDAL 4.0 + NumPy 2.5) ---
from osgeo import gdal
gdal.DontUseExceptions()  # фикс FutureWarning "Neither gdal.UseExceptions... "
warnings.filterwarnings("ignore", message="Setting the shape on a NumPy array")
warnings.filterwarnings("ignore", category=DeprecationWarning)  # шум сторонних lib (laspy/numpy) в демо\nwarnings.filterwarnings("ignore", message="Neither gdal.UseExceptions.*", category=FutureWarning)

backend = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')
for pkg in ['packages/sima-dem-core/src', 'packages/sima-dem-ground/src',
            'packages/sima-dem-dsm/src', 'packages/sima-relief-service/src']:
    sys.path.insert(0, str(backend / pkg))

import numpy as np
import laspy
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from osgeo import ogr
from sima_relief_service import (
    ReliefService, ReliefRequest, ReliefParams, TileInput,
    DerivativesParams, VectorsParams, HeightsParams, SmrfParams, SmoothingParams,
    assess_materials,
)
print('Импорт готов')

## 1. Датасет и пути

Переключатель `DATASET = 'demo' | 'test'` (как в `sima_dsm_demo.ipynb`).
Для `demo` доступны эталонные выходы legacy-конвейера для верификации.

In [ ]:
DATASET = 'demo'

if DATASET == 'demo':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/pt000100.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/00000100.tif'
    REF_DIR = Path(LAS_PATH).parent
elif DATASET == 'test':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_ground_TLO.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g.tif'
    REF_DIR = Path(TIF_PATH).parent

OUTPUT_DIR = str(backend / 'output' / f'relief_service_notebook_{DATASET}')
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESOLUTION = 1.0

# CRS — единый источник истины (сам TIFF)
with rasterio.open(TIF_PATH) as src:
    CRS = src.crs.to_wkt()
    AFS_BOUNDS = src.bounds
print(f'Датасет: {DATASET} | CRS: {CRS[:60]}... | разрешение АФС из тайла')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

def read_raster(path):
    """Прочитать растр как float-массив с nodata→nan + bounds (без read_masks)."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
        return arr, src.bounds

## 2. Оценка материалов (Q3 «Оценка файлов ВЛС/АФС»)

Извлекаем: СК, площадь экстента, разрешение (АФС), плотность точек и диапазон
высот ТЛО (ВЛС). Масштаб ОФП/ТЛО — параметр съёмки (не из файла).

In [ ]:
assessment = assess_materials(vls_files=[LAS_PATH], afs_files=[TIF_PATH])
vls, afs = assessment.vls, assessment.afs
print('=== ВЛС (LAS) ===')
print(f'  СК задана: {bool(vls.crs)} | площадь: {vls.extent_area_km2:.4f} км²')
print(f'  плотность: {vls.density_pts_m2:.2f} pts/m² | высоты ТЛО: {vls.tlo_height_range_m}')
print('=== АФС (TIFF) ===')
print(f'  разрешение: {afs.resolution_m:.3f} м | площадь: {afs.extent_area_km2:.4f} км²')
print(f'  тайлов: {afs.tiles_total}, ok: {afs.tiles_ok}, failed: {afs.tiles_failed}')

In [ ]:
# Визуализация: облако ТЛО (ВЛС) — план с раскраской по Z + гистограмма высот
las = laspy.read(LAS_PATH)
x = np.asarray(las.x); y = np.asarray(las.y); z = np.asarray(las.z)
n = len(z)
step = max(1, n // 20000)  # даунсэмплинг для скорости
xs, ys, zs = x[::step], y[::step], z[::step]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
sc = ax.scatter(xs, ys, c=zs, s=1, cmap='terrain')
ax.set_aspect('equal'); ax.set_title(f'Облако ТЛО ({n:,} точек, показан каждый {step})')
plt.colorbar(sc, ax=ax, shrink=0.7, label='Z, м')
ax = axes[1]
ax.hist(z, bins=100, color='steelblue', edgecolor='none', alpha=0.85)
ax.set_title('Распределение высот Z'); ax.set_xlabel('Z, м'); ax.set_ylabel('кол-во точек')
plt.tight_layout(); plt.savefig(str(Path(OUTPUT_DIR)/'assessment_vls.png'), dpi=140); plt.show()

## 3. Запуск сервиса рельефа

`ReliefService.run` — оркестрация шагов по тайлам: crop → filter → ЦМР → smooth →
derivatives → vectors → heights. Трекаются статусы шагов; упавший тайл → failed + причина.

In [ ]:
params = ReliefParams(
    target_crs=CRS, filter_method='smrf',
    smrf=SmrfParams(slope=0.2, window=16, threshold=0.45, scalar=1.2),
    smoothing=SmoothingParams(enabled=True, sigma=1.0, order=0, window=5),
    derivatives=DerivativesParams(slopes=True, slopes_res=RESOLUTION,
                                  aspect=True, aspect_res=RESOLUTION,
                                  tpi=True, interpolation=True, inter_amp=100),
    vectors=VectorsParams(horizontals=[0.5, 2.0, 5.0], tin=True),
    heights=HeightsParams(enabled=True, source='dem', step=10),
    deterministic=True, seed=42,
)
request = ReliefRequest(params=params, project_id='demo_project', resolution=RESOLUTION,
                       tiles=[TileInput(name='pt000100', vls_path=LAS_PATH, afs_path=TIF_PATH)])
svc = ReliefService(root_dir=OUTPUT_DIR)
result = svc.run(request)
job = result.job
tile = job.tiles[0]
print(f'Job: {job.status} | сессия: {job.session_id}')
print(f'Тайлов: {job.tiles_total}, done: {job.tiles_done}, failed: {job.tiles_failed}, прогресс: {job.progress}%')

In [ ]:
print(f'Тайл {tile.name}: {tile.status} (id={tile.id})')
print('Шаги:')
for s in tile.steps:
    msg = f' — {s.message}' if s.message else ''
    print(f'  {s.name:10} {s.status:8}{msg}')
print('Артефакты (Q3 форматы):')
for a in tile.output_files:
    print(f'  [{a.kind:7}] {a.layer:12} {Path(a.path).name}  ({a.size_bytes or 0} байт)')

art = {a.layer: a.path for a in tile.output_files}

## 4. Визуализация результатов

Сетка растров: ЦМР, сглаженная ЦМР, уклон, экспозиция, TPI.

In [ ]:
dtm, _ = read_raster(art['dtm'])
smooth, _ = read_raster(art.get('dtm_smooth', art['dtm']))
slope, _ = read_raster(art['slope'])
aspect, _ = read_raster(art['aspect'])
tpi, _ = read_raster(art['tpi'])

zmin = np.nanmin([np.nanmin(dtm), np.nanmin(smooth)])
zmax = np.nanmax([np.nanmax(dtm), np.nanmax(smooth)])
tnorm = mcolors.Normalize(vmin=zmin, vmax=zmax)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
panels = [('ЦМР (DTM)', dtm, 'terrain', tnorm),
          ('Сглаженная ЦМР', smooth, 'terrain', tnorm),
          ('Уклон (°)', slope, 'hot', None),
          ('Экспозиция (°)', aspect, 'hsv', None),
          ('TPI', tpi, 'RdBu_r', None)]
for ax, (title, data, cmap, norm) in zip(axes.flatten(), panels):
    im = ax.imshow(data, cmap=cmap, norm=norm)
    ax.set_title(title, fontsize=14); plt.colorbar(im, ax=ax, shrink=0.7)
axes.flatten()[5].set_visible(False)
plt.suptitle(f'СИМА — {DATASET} (resolution={RESOLUTION}m)', fontsize=16)
plt.tight_layout(); plt.savefig(str(Path(OUTPUT_DIR)/'overview.png'), dpi=140); plt.show()

### 4.1 Горизонтали и отметки высот поверх ЦМР

Горизонтали (.shp) и отметки высот (.shp) накладываются на ЦМР в мировых координатах.

In [ ]:
def read_lines(path):
    ds = ogr.Open(path); lyr = ds.GetLayer(0); lines = []
    for f in lyr:
        g = f.GetGeometryRef()
        if g.GetGeometryType() in (2, 2+0x80000000):  # wkbLineString(25D)
            lines.append(np.array(g.GetPoints())[:, :2])
    return lines

def read_points(path):
    ds = ogr.Open(path); lyr = ds.GetLayer(0); pts = []
    for f in lyr:
        g = f.GetGeometryRef(); pts.append((g.GetX(), g.GetY(), g.GetZ()))
    return np.array(pts)

with rasterio.open(art['dtm']) as src:
    west, east = src.bounds.left, src.bounds.right
    south, north = src.bounds.bottom, src.bounds.top

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(dtm, cmap='terrain', norm=tnorm, extent=[west, east, south, north])
for interval in [0.5, 2.0, 5.0]:
    c = art.get('contours')
# берём один файл горизонталей для показа (последний интервал)
import glob
shp_files = sorted(glob.glob(str(Path(tile.output_dir) / '*_contours_*.shp')))
for shp in shp_files:
    for ln in read_lines(shp):
        ax.plot(ln[:, 0], ln[:, 1], color='k', linewidth=0.4, alpha=0.6)
hp = art.get('heights')
if hp:
    pts = read_points(hp)
    ax.scatter(pts[:, 0], pts[:, 1], c='red', s=4, zorder=5, label='отметки высот')
    ax.legend()
ax.set_title('ЦМР + горизонтали (.shp) + отметки высот (.shp)'); ax.set_aspect('equal')
plt.colorbar(im, ax=ax, shrink=0.6, label='Z, м')
plt.tight_layout(); plt.savefig(str(Path(OUTPUT_DIR)/'contours_heights.png'), dpi=140); plt.show()

### 4.2 TIN-поверхность

Вершины TIN — ground-точки (показаны точки; сама TIN-поверхность экспортирована в .dxf).

In [ ]:
gl = art.get('ground_las')
if gl:
    g = laspy.read(gl)
    gx, gy, gz = np.asarray(g.x), np.asarray(g.y), np.asarray(g.z)
    st = max(1, len(gx) // 5000)
    fig, ax = plt.subplots(figsize=(11, 9))
    sc = ax.scatter(gx[::st], gy[::st], c=gz[::st], s=2, cmap='terrain')
    ax.set_aspect('equal'); ax.set_title(f'Вершины TIN (ground: {len(gx):,} точек)\nповерхность → {Path(art["tin"]).name}')
    plt.colorbar(sc, ax=ax, shrink=0.6, label='Z, м')
    plt.tight_layout(); plt.savefig(str(Path(OUTPUT_DIR)/'tin_vertices.png'), dpi=140); plt.show()
else:
    print('ground_las не сохранён')

## 5. Верификация против эталонных выходов (demo_data)

Сравнение ЦМР/уклон/экспозиция/TPI сервиса с эталонами legacy-конвейера.

In [ ]:
def raster_stats(p):
    with rasterio.open(p) as src:
        a = src.read(1, masked=True)  # без read_masks → нет NumPy DeprecationWarning
        v = a.compressed()
        return float(v.min()), float(v.max()), float(v.mean())

if DATASET == 'demo':
    pairs = [('dtm', REF_DIR / 'pt000100_dem.tif'),
             ('slope', REF_DIR / 'pt000100_slope.tif'),
             ('aspect', REF_DIR / 'pt000100_aspect.tif'),
             ('tpi', REF_DIR / 'pt000100_tpi.tif')]
    print(f'{"слой":8} {"сервис min/max/mean":28} {"эталон min/max/mean":28}')
    for layer, ref in pairs:
        got = art.get(layer)
        if got and Path(ref).exists():
            gg, rr = raster_stats(got), raster_stats(str(ref))
            print(f'{layer:8} {gg[0]:7.2f}/{gg[1]:7.2f}/{gg[2]:7.2f}   {rr[0]:7.2f}/{rr[1]:7.2f}/{rr[2]:7.2f}')
        else:
            print(f'{layer:8} — нет выходного или эталона')
else:
    print('Верификация против эталона доступна только для DATASET=\'demo\'')

In [ ]:
# Визуализация верификации: построено vs эталон + карта разницы (для ЦМР)
if DATASET == 'demo' and (REF_DIR / 'pt000100_dem.tif').exists():
    ref, _ = read_raster(str(REF_DIR / 'pt000100_dem.tif'))
    h = min(dtm.shape[0], ref.shape[0]); w = min(dtm.shape[1], ref.shape[1])
    b, r = dtm[:h, :w], ref[:h, :w]
    valid = np.isfinite(b) & np.isfinite(r)
    diff = np.where(valid, b - r, np.nan)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, title, data in [(axes[0], 'Построено', b), (axes[1], 'Эталон', r)]:
        im = ax.imshow(data, cmap='terrain', norm=tnorm); ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.7)
    im = axes[2].imshow(diff, cmap='RdBu_r', vmin=-5, vmax=5)
    axes[2].set_title(f'Разница ΔZ (mean={np.nanmean(diff):.2f} м)')
    plt.colorbar(im, ax=axes[2], shrink=0.7)
    plt.tight_layout(); plt.savefig(str(Path(OUTPUT_DIR)/'verify_dtm.png'), dpi=140); plt.show()

## 6. Повторный запуск тайла — история не затирается

При повторе создаётся новый `tile_id` (старые выходные данные остаются), REFINEMENT_PLAN Г1.

In [ ]:
result2 = svc.run(request)
t1, t2 = result.job.tiles[0], result2.job.tiles[0]
print(f'Запуск 1: id={t1.id}')
print(f'Запуск 2: id={t2.id}')
print(f'ID различаются (история не затёрта): {t1.id != t2.id}')
print(f'Старая сессия сохранена: {Path(t1.output_dir).exists()}')

## 7. Выходные файлы

In [ ]:
print(f'Выходные файлы сессии ({job.output_dir}):')
for f in sorted(Path(job.output_dir).rglob('*')):
    if f.is_file() and f.suffix in ('.tif', '.las', '.shp', '.dxf', '.shx', '.dbf', '.prj'):
        size = f.stat().st_size / 1024
        print(f'  {f.relative_to(job.output_dir)}  ({size:.1f} KB)')

## Итог

`sima-relief-service` — отдельная библиотека сервисного слоя рельефа: работает с
LAS/TIFF, извлекает метаданные материалов, корректно считает ЦМР и производные,
выводит форматы Q3 (.geotiff/.shp/.dxf), ведёт сессии и статусы по тайлам, готова
к обёртке в реальный backend-сервис (FastAPI/Celery/S3).